In [1]:
# Interest Calculator — simple, compound, and continuous
# To re-run after quitting: delete empty cells below, then Restart Kernel
import tkinter as tk
import tkinter.ttk as ttk
import math
from functools import partial

def main():
    root = tk.Tk()
    root.title('Interest Calculator')
    root.lift()
    root.attributes('-topmost', True)

    # Grid row constants
    ROW_PRINCIPAL   = 1
    ROW_RATE        = 2
    ROW_MODE        = 3
    ROW_FREQ        = 4
    ROW_TIME        = 5
    ROW_COMPUTE_BTN = 6
    ROW_RESULT      = 7
    ROW_ERROR       = 8
    ROW_BUTTONS     = 11

    result_label = None
    result_value = None
    error_label  = None

    def show_error(message):
        nonlocal result_label, result_value, error_label
        for widget in (result_label, result_value, error_label):
            if widget is not None:
                widget.destroy()
        result_label = result_value = None
        error_label = tk.Label(root, text=message, fg='red')
        error_label.grid(row=ROW_ERROR, column=1, columnspan=2,
                         padx=3, pady=5, sticky='w')

    def show_result(value):
        nonlocal result_label, result_value, error_label
        for widget in (result_label, result_value, error_label):
            if widget is not None:
                widget.destroy()
        error_label = None
        result_label = tk.Label(root, text="The final principal is:")
        result_label.grid(row=ROW_RESULT, column=1, padx=3, pady=5, sticky='w')
        result_value = tk.Label(root, text=f"${value:,.2f}",
                                relief='sunken', width=15, anchor='w')
        result_value.grid(row=ROW_RESULT, column=2, padx=3, sticky='w')

    def on_mode_change(*_):
        if mode.get() == 'compound':
            freq_label.grid()
            q3.grid()
        else:
            freq_label.grid_remove()
            q3.grid_remove()

    def compute_principal(w, x, y1, mode_var, z):
        try:
            orig_prin = w.get()
        except tk.TclError:
            show_error("Error: Original principal must be a number.")
            return
        try:
            rate = x.get() / 100
        except tk.TclError:
            show_error("Error: Interest rate must be a number.")
            return
        try:
            time = z.get()
        except tk.TclError:
            show_error("Error: Time must be a number.")
            return
        if orig_prin < 0:
            show_error("Error: Principal cannot be negative.")
            return
        if time < 0:
            show_error("Error: Time cannot be negative.")
            return

        interest_mode = mode_var.get()
        if interest_mode == 'compound':
            try:
                freq = y1.get()
            except tk.TclError:
                show_error("Error: Compounding frequency must be a number.")
                return
            if freq <= 0:
                show_error("Error: Compounding frequency must be greater than 0.")
                return
            fin_prin = orig_prin * (1 + (rate / freq)) ** (freq * time)
        elif interest_mode == 'continuous':
            fin_prin = orig_prin * math.exp(rate * time)
        else:
            fin_prin = orig_prin * (1 + rate * time)

        show_result(round(fin_prin, 2))

    def clear():
        nonlocal result_label, result_value, error_label
        for widget in (result_label, result_value, error_label):
            if widget is not None:
                widget.destroy()
        result_label = result_value = error_label = None
        q1.delete(0, tk.END);  w.set(0.0)
        q2.delete(0, tk.END);  x.set(0.0)
        q3.delete(0, tk.END);  y1.set(0.0)
        q5.delete(0, tk.END);  z.set(0.0)
        mode.set('compound')
        on_mode_change()

    def quit_app():
        root.destroy()

    # --- UI Layout ---
    tk.Label(root, text="Original principal ($):").grid(
        row=ROW_PRINCIPAL, column=1, padx=3, sticky='w')
    w = tk.DoubleVar()
    q1 = tk.Entry(root, textvariable=w)
    q1.grid(row=ROW_PRINCIPAL, column=2, padx=5, pady=3, sticky='w')

    tk.Label(root, text="Annual interest percentage:").grid(
        row=ROW_RATE, column=1, padx=3, sticky='w')
    x = tk.DoubleVar()
    q2 = tk.Entry(root, textvariable=x)
    q2.grid(row=ROW_RATE, column=2, padx=5, pady=3, sticky='w')

    tk.Label(root, text="Interest type:").grid(
        row=ROW_MODE, column=1, padx=3, sticky='w')
    mode = tk.StringVar(value='compound')
    mode.trace_add('write', on_mode_change)
    mode_frame = tk.Frame(root)
    mode_frame.grid(row=ROW_MODE, column=2, padx=5, pady=3, sticky='w')
    tk.Radiobutton(mode_frame, text='Compound',   variable=mode, value='compound').pack(side='left')
    tk.Radiobutton(mode_frame, text='Continuous', variable=mode, value='continuous').pack(side='left', padx=6)
    tk.Radiobutton(mode_frame, text='Simple',     variable=mode, value='simple').pack(side='left')

    freq_label = tk.Label(root, text="Compounding frequency per year:")
    freq_label.grid(row=ROW_FREQ, column=1, padx=3, sticky='w')
    y1 = tk.DoubleVar()
    q3 = tk.Entry(root, textvariable=y1)
    q3.grid(row=ROW_FREQ, column=2, padx=5, pady=3, sticky='w')

    tk.Label(root,
             text="Time that interest accrues (years):\n"
                  " (fractions are allowed, e.g. 1.25)",
             justify='left').grid(
        row=ROW_TIME, column=1, padx=3, sticky='w')
    z = tk.DoubleVar()
    q5 = tk.Entry(root, textvariable=z)
    q5.grid(row=ROW_TIME, column=2, padx=5, pady=3, sticky='w')

    principal_cmd = partial(compute_principal, w, x, y1, mode, z)
    button1 = tk.Button(root, text="Compute final principal", command=principal_cmd)
    button1.grid(row=ROW_COMPUTE_BTN, column=1, padx=3, pady=7)

    button2 = tk.Button(root, text='Clear', command=clear)
    button2.grid(row=ROW_BUTTONS, column=1, pady=5)

    button3 = tk.Button(root, text='Quit', command=quit_app)
    button3.grid(row=ROW_BUTTONS, column=2, padx=3, pady=5, sticky='w')

    root.mainloop()

main()